# 11 — Rule-based entity extraction

Run the versioned offline extractor on 1,000 real A2 training CVs and 1,000 real A2 training JDs. The seed vocabulary supports `KNOWLEDGE`, `SKILL`, `TITLE` and `DEGREE`; it is deliberately limited and carries its own canonical namespace, never ESCO IDs.

The package emits field-relative half-open evidence offsets and deterministic IDs. A mention does not establish proficiency, applied experience or satisfaction of a job requirement. Context status is a bounded heuristic. Real records remain in ignored local artifacts. The bounded preview below shows only redacted seed-vocabulary mentions; full source fields are never printed.

In [1]:
import json
import pandas as pd
from candidate_screener.config import PROJECT_ROOT

# Aggregate outputs only: cap every saved output at 2,000 characters.
def show_summary(value):
    rendered = json.dumps(value, indent=2, default=str)
    print(rendered if len(rendered) <= 1900 else rendered[:1850] + "\n[output truncated]")

SAMPLE_SIZE = 1000  # per document type, exclusively from the frozen A2 train region
SEED = 0


In [2]:
from candidate_screener.extraction.run import run_extraction

result = run_extraction(sample_size=SAMPLE_SIZE, seed=SEED)
show_summary(result["summary"]["input_counts"])


{
  "cv": {
    "raw_documents": 210250,
    "eligible_documents": 157688,
    "sampled_documents": 1000,
    "fields": [
      "Position",
      "Moreinfo",
      "Looking For",
      "Highlights"
    ]
  },
  "jd": {
    "raw_documents": 141897,
    "eligible_documents": 106423,
    "sampled_documents": 1000,
    "fields": [
      "Position",
      "Long Description"
    ]
  }
}


In [3]:
show_summary(result["summary"]["coverage"]["cv"])

{
  "documents": 1000,
  "documents_with_entities": 680,
  "descriptive_coverage": 0.68,
  "entities": 2994,
  "by_label": {
    "DEGREE": 32,
    "KNOWLEDGE": 2214,
    "SKILL": 199,
    "TITLE": 549
  },
  "documents_with_label": {
    "DEGREE": 30,
    "KNOWLEDGE": 521,
    "SKILL": 123,
    "TITLE": 338
  },
  "by_context_status": {
    "learning": 148,
    "negated": 1,
    "unqualified": 2845
  }
}


In [4]:
show_summary(result["summary"]["coverage"]["jd"])

{
  "documents": 1000,
  "documents_with_entities": 754,
  "descriptive_coverage": 0.754,
  "entities": 3604,
  "by_label": {
    "DEGREE": 41,
    "KNOWLEDGE": 3023,
    "SKILL": 205,
    "TITLE": 335
  },
  "documents_with_label": {
    "DEGREE": 36,
    "KNOWLEDGE": 662,
    "SKILL": 137,
    "TITLE": 229
  },
  "by_context_status": {
    "learning": 9,
    "negated": 4,
    "unqualified": 3591
  }
}


In [5]:
show_summary({"integrity": result["summary"]["integrity"], "gold_accuracy": result["summary"]["gold_accuracy"]})

{
  "integrity": {
    "entities_checked": 6598,
    "unique_entity_ids": true,
    "exact_field_offsets": true,
    "training_only": true
  },
  "gold_accuracy": {
    "status": "blocked",
    "reason": "Gold adapters pending; unlabelled A2 measures descriptive coverage only"
  }
}


## Evidence field dictionary

| Field | Meaning |
|---|---|
| `entity_id` | Deterministic identifier for this evidence record |
| `doc_id` | Original corpus document identifier |
| `source` | Corpus provenance, e.g. Djinni CV or JD |
| `doc_type` | `cv` or `jd` |
| `field` | Original source field containing the evidence |
| `text_version` | Version identifier for the immutable source text |
| `start` | Inclusive character offset in the original field |
| `end` | Exclusive character offset in the original field |
| `label` | KNOWLEDGE, SKILL, TITLE or DEGREE |
| `surface` | Exact original field slice matched by the extractor |
| `canonical_id` | Seed-vocabulary identifier, nullable; never an ESCO claim |
| `method` | Extraction method, here `seed-rule` |
| `extractor_version` | Version of rules producing this record |
| `confidence` | Null for rules; no calibrated certainty is claimed |
| `context_status` | Heuristic negated, learning or unqualified context |

Offsets describe the **original** text. Contact redaction may change the displayed surface; displayed strings must never be used to revalidate or reinterpret offsets.

## Five actual extracted mentions

This is a bounded view of real extraction records. The display helper admits only the reviewed seed phrases and applies contact redaction; `assert_clean` runs here before saving any output. Entire source fields, document IDs and unreviewed neural predictions are omitted. Regex redaction is not a general name anonymizer.

In [6]:
from pathlib import Path
from candidate_screener.extraction.display import preview_entities
from candidate_screener.annotation.redact import assert_clean

real_entities = pd.read_parquet(Path(result["run_dir"]) / "entities.parquet").to_dict("records")
real_preview = preview_entities(real_entities, limit=5)
assert_clean(pd.Series([row["surface"] for row in real_preview], dtype="string"), "reviewed rule entity preview")
show_summary(real_preview)


[
  {
    "surface": "project management",
    "label": "SKILL",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  },
  {
    "surface": "Project Manager",
    "label": "TITLE",
    "field": "Position",
    "context_status": "unqualified",
    "doc_type": "cv"
  },
  {
    "surface": "Java",
    "label": "KNOWLEDGE",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  },
  {
    "surface": "Python",
    "label": "KNOWLEDGE",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  },
  {
    "surface": ".NET",
    "label": "KNOWLEDGE",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  }
]


## Interpretation

Coverage is **descriptive on unlabelled data**. Counts describe the vocabulary and rules as much as the documents. Negated and learning/aspirational mentions are retained with context rather than turned into positive screening evidence. Rules provide null confidence because they have no calibrated probabilities.

Job titles and qualification phrases preserve the matched surface; they do not infer complete career history, institutions, credential equivalence or experience totals. Unknown mappings remain null. The committed baseline keyword helper and frozen evaluation/annotation records are unchanged.

## Illustrative synthetic evidence

The following example is authored text, not a resume. Redaction and `assert_clean` run **inside this notebook** before any synthetic display. Offsets are checked against the immutable original synthetic field; displayed redaction is a separate operation.

In [7]:
from candidate_screener.annotation.redact import redact_text, assert_clean
from candidate_screener.extraction import extract_document

synthetic_text = "No Python experience. Learning Kubernetes. C++ and C# development with .NET."
synthetic_document = {
    "doc_id": "synthetic-demo", "source": "synthetic", "doc_type": "cv",
    "fields": {"Moreinfo": synthetic_text},
}
synthetic_entities = extract_document(synthetic_document)
assert all(synthetic_text[e["start"]:e["end"]] == e["surface"] for e in synthetic_entities)
synthetic_display = pd.DataFrame(synthetic_entities).loc[:, ["surface", "label", "start", "end", "context_status"]].head(5)
synthetic_display["surface"] = synthetic_display["surface"].map(redact_text)
assert_clean(synthetic_display["surface"], "synthetic notebook example")
show_summary(synthetic_display.to_dict(orient="records"))


[
  {
    "surface": "Python",
    "label": "KNOWLEDGE",
    "start": 3,
    "end": 9,
    "context_status": "negated"
  },
  {
    "surface": "Kubernetes",
    "label": "KNOWLEDGE",
    "start": 31,
    "end": 41,
    "context_status": "learning"
  },
  {
    "surface": "C++",
    "label": "KNOWLEDGE",
    "start": 43,
    "end": 46,
    "context_status": "unqualified"
  },
  {
    "surface": "C#",
    "label": "KNOWLEDGE",
    "start": 51,
    "end": 53,
    "context_status": "unqualified"
  },
  {
    "surface": ".NET",
    "label": "KNOWLEDGE",
    "start": 71,
    "end": 75,
    "context_status": "unqualified"
  }
]


## Local artifacts and reruns

The package writes evidence parquet, a sample manifest and run metadata under `data/processed/extraction/`. Provenance includes source/partition digests, sampling parameters, selected IDs/fields and extractor/vocabulary versions. Compatible reruns are deterministic; incompatible artifacts are refused rather than overwritten.

Notebook 12 validates integrity and reproducibility separately from supervised accuracy. Extraction alone has not established a Recall@10 improvement or production readiness.